# Season-Stratified Analysis
## School term versus vacation WITHIN each season

Interpretation requires comparing calendar regimes within the same season, and seasons within the same regime, rather than pooling all school-term dates against all vacation dates.

The rationale is that a school-calendar association should be examined within comparable seasonal conditions. If a pooled difference is mainly seasonal, it should shrink after this adjustment. Stratification is a diagnostic, not proof that meteorological confounding has been removed.

**Data access.** The SIMA measurements used here are not
redistributed in this repository (SIMA provided them for this study on the condition
that they are not disseminated); the repository holds the code, the official school
calendar and all derived results. This notebook reads files that notebooks 00 and 01
build from those measurements, and its configuration cell stops with a clear message
if they are missing. To rebuild them, request the annual workbooks from SIMA, place
them in `data_raw/` and run notebook 00 (its header gives the exact file format).

**Role in the analysis.** Diagnostic and sensitivity: season-stratified contrasts with month fixed effects. The analytical hierarchy of the whole repository is stated in the README.

## 1. Initial setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

contaminantes = ['CO', 'NO2', 'O3', 'PM10', 'PM2.5']
colores_contaminantes = {'CO': '#3498db', 'NO2': '#e74c3c', 'O3': '#2ecc71',
                         'PM10': '#f39c12', 'PM2.5': '#9b59b6'}
orden_temporadas = ['invierno', 'primavera', 'verano', 'otono']
etiquetas_temporada = {'invierno': 'Winter (DJF)', 'primavera': 'Spring (MAM)',
                       'verano': 'Summer (JJA)', 'otono': 'Autumn (SON)'}

output_dir = Path('../data_processed')
estacional_dir = output_dir / 'analisis_estacional'
estacional_dir.mkdir(exist_ok=True)

print('Libraries imported successfully')

# Check required inputs (restricted data; see heading)
# Input-data check (restricted data; see heading)
archivos_requeridos = [output_dir / 'panel_pico_hibrido.csv',
                       output_dir / 'panel_pico_MIMA.csv',
                       output_dir / 'gaps_franjas_hibrido.csv']
faltantes = [str(a) for a in archivos_requeridos if not a.exists()]
if faltantes:
    raise FileNotFoundError(
        ''
        ''
        "Files derived from the SIMA measurements are missing; they are not redistributed in this repository. "
        "Request the annual workbooks from SIMA, place them in data_raw/ and run notebooks 00 and 01. "
        f"Missing: {faltantes}")
print(f"Input data found: {len(archivos_requeridos)} files")

Libraries imported successfully
Input data found: 3 files


## 2. Loading the panels from notebook 01

In [2]:
panel = pd.read_csv('../data_processed/panel_pico_hibrido.csv', parse_dates=['date'])
panel_mima = pd.read_csv('../data_processed/panel_pico_MIMA.csv', parse_dates=['date'])
gaps = pd.read_csv('../data_processed/gaps_franjas_hibrido.csv', parse_dates=['date'])

print("=" * 70)
print('PANELS LOADED')
print("=" * 70)
print(f"Arrival-window panel (hybrid): {panel.shape[0]:,} day-unit rows")
print(f"Arrival-window panel (MIMA): {panel_mima.shape[0]:,} rows")
print(f"Within-day gap panel: {gaps.shape[0]:,} rows")
print(f"\nDays by season x regime:")
print(panel.groupby(['temporada4', 'clase'])['date'].nunique().unstack())

PANELS LOADED
Arrival-window panel (hybrid): 6,576 day-unit rows
Arrival-window panel (MIMA): 6,576 rows
Within-day gap panel: 6,576 rows

Days by season x regime:
clase           0      1
temporada4              
invierno     58.0  213.0
otono         NaN  273.0
primavera    36.0  240.0
verano      118.0  158.0


## 3. Contrast function with weekly-clustered errors

We use the same estimator as the rest of the analysis: OLS on log concentrations with monitoring-unit, MONTH and day-of-week fixed effects, clustering errors by ISO week with the same finite-sample correction. Within a season, month fixed effects adjust for school-term dates being concentrated in particular months, such as one end of the summer period.

### The contrast equation

Within each stratum, the outcome is either the log concentration or the within-day gap:

$$ y_{st}=\alpha+\beta\,\text{School}_t+\eta_s+\gamma_{m(t)}+\delta_{d(t)}+\varepsilon_{st}. $$

Here $\eta_s$ denotes monitoring-unit fixed effects, $\gamma_{m(t)}$ month effects and $\delta_{d(t)}$ day-of-week effects. Because the outcome is on a log scale, the estimate is reported as a percentage:

$$ \%\Delta=(e^\beta-1)\times100,\qquad \text{95% CI: }(e^{\beta\pm1.96SE}-1)\times100. $$

This is a confidence interval for $\beta$ transformed to percentages: construct the interval on the coefficient scale, where it is symmetric, then transform its endpoints. Errors are clustered by ISO week because consecutive days share persistent weather and other conditions. Treating them as independent would tend to understate uncertainty. Notebook 05 gives the clustered covariance formula.

In [3]:
def contraste_clase(df, columna_y, con_controles=True):
    "\n    Estimate the 'clase' coefficient for columna_y with weekly-clustered errors.\n    Return beta, percent effect, 95% CI, p-value and sample sizes.\n    "
    df = df.dropna(subset=[columna_y]).copy()
    piezas = [pd.Series(1.0, index=df.index, name='const'), df['clase'].astype(float)]
    if con_controles:
        piezas.append(pd.get_dummies(df['estacion'], prefix='est', drop_first=True).astype(float))
        piezas.append(pd.get_dummies(df['mes'], prefix='mes', drop_first=True).astype(float))
        piezas.append(pd.get_dummies(df['dow'], prefix='dow', drop_first=True).astype(float))
    X = pd.concat(piezas, axis=1)
    Xv = X.values
    n, k = Xv.shape
    if n <= k or df['clase'].nunique() < 2:
        return None
    XtX_inv = np.linalg.inv(Xv.T @ Xv)
    y = df[columna_y].values
    beta_todos = XtX_inv @ Xv.T @ y
    residuos = y - Xv @ beta_todos

    grupos = pd.factorize(df['wk'])[0]
    orden = np.argsort(grupos, kind='stable')
    Xo, ro, go = Xv[orden], residuos[orden], grupos[orden]
    inicios = np.flatnonzero(np.r_[True, go[1:] != go[:-1]])
    G = len(inicios)
    S = np.zeros((k, k))
    fines = np.r_[inicios[1:], len(go)]
    for s, e in zip(inicios, fines):
        zg = Xo[s:e].T @ ro[s:e]
        S += np.outer(zg, zg)
    V = XtX_inv @ S @ XtX_inv
    V *= (G / (G - 1)) * ((n - 1) / (n - k))

    idx_clase = X.columns.get_loc('clase')
    beta = beta_todos[idx_clase]
    se = np.sqrt(V[idx_clase, idx_clase])
    z = beta / se
    return {'beta': beta, 'se': se,
            'pct': (np.exp(beta) - 1) * 100,
            'ci_lo_pct': (np.exp(beta - 1.96 * se) - 1) * 100,
            'ci_hi_pct': (np.exp(beta + 1.96 * se) - 1) * 100,
            'p': 2 * (1 - stats.norm.cdf(abs(z))),
            'n': n, 'G_semanas': G,
            'n_clase': int((df['clase'] == 1).sum()), 'n_vac': int((df['clase'] == 0).sum())}

print('Contrast function ready')

Contrast function ready


## 4. Seasonal pattern: monthly levels by regime

Each pollutant follows its annual cycle. The school-term and vacation curves allow comparison of those cycles in months where both regimes occur.

In [4]:
nivel_mensual = (panel.groupby(['mes', 'clase'])[contaminantes].mean().reset_index())
conteo_mensual = panel.groupby(['mes', 'clase'])['date'].nunique().rename('n_dias').reset_index()
nivel_mensual = nivel_mensual.merge(conteo_mensual, on=['mes', 'clase'])
# Omit plotted group means with fewer than 10 days
nivel_mensual_grafica = nivel_mensual[nivel_mensual['n_dias'] >= 10]

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)
axes = axes.flatten()

for i, c in enumerate(contaminantes):
    ax = axes[i]
    for cl, color, etiqueta in [(0, '#3498db', 'Vacation'), (1, '#e74c3c', 'School term')]:
        sub = nivel_mensual_grafica[nivel_mensual_grafica['clase'] == cl]
        ax.plot(sub['mes'], sub[c], marker='o', color=color, label=etiqueta, linewidth=2)
    ax.set_title(c, fontsize=12, fontweight='bold')
    ax.set_xticks(range(1, 13))
    if i >= 3:
        ax.set_xlabel('Month', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Mean level (arrival window)', fontsize=11, fontweight='bold')
axes[3].set_ylabel('Mean level (arrival window)', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=10)
axes[5].axis('off')
axes[5].text(0.05, 0.6, 'Reading the plot: both lines follow\na similar seasonal pattern.\n\nWhere they overlap, the regime\ndifference is smaller than much\nof the seasonal variation.\n\n(Month-regime means with fewer\nthan 10 days are not plotted.)',
             fontsize=11, va='top')
fig.suptitle('Monthly pollutant levels: school-term vs vacation dates',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(estacional_dir / '01_patron_mensual_por_regimen.png', dpi=150, bbox_inches='tight')
plt.show()

nivel_mensual.to_csv(estacional_dir / 'nivel_mensual_por_regimen.csv', index=False)
conteo_mensual.to_csv(estacional_dir / 'n_dias_mensual_por_regimen.csv', index=False)
print(f"Saved to: {estacional_dir / 'nivel_mensual_por_regimen.csv'}")
print(f"Saved to: {estacional_dir / 'n_dias_mensual_por_regimen.csv'}")
print(f"Plot saved to: {estacional_dir / '01_patron_mensual_por_regimen.png'}")

Saved to: ../data_processed/analisis_estacional/nivel_mensual_por_regimen.csv
Saved to: ../data_processed/analisis_estacional/n_dias_mensual_por_regimen.csv
Plot saved to: ../data_processed/analisis_estacional/01_patron_mensual_por_regimen.png


## 5. Exploration: levels by season and regime using boxplots

Comparing all school-term dates against all vacation dates mixes seasons. Here the adjusted contrast, with monitoring-unit, month and day-of-week controls and weekly-clustered errors, is repeated within each season. Autumn cannot be estimated separately because the official calendar contains no autumn vacation dates in the study period.

This section is DESCRIPTIVE. Boxplots summarize distributions without implying a causal interpretation. The adjusted contrast table is retained below because it informs interpretation, while the formal results with confidence intervals and clustered errors appear in Section 7 and notebooks 05 and 06.

In [5]:
resultados_estratos = []
for c in contaminantes:
    fila_global = contraste_clase(panel, f'l_{c}')
    fila_global.update({'contaminante': c, 'estrato': 'global'})
    resultados_estratos.append(fila_global)
    for temporada in orden_temporadas:
        sub = panel[panel['temporada4'] == temporada]
        resultado = contraste_clase(sub, f'l_{c}')
        if resultado is None:
            continue
        resultado.update({'contaminante': c, 'estrato': temporada})
        resultados_estratos.append(resultado)

tabla_estratos = pd.DataFrame(resultados_estratos)[
    ['contaminante', 'estrato', 'pct', 'ci_lo_pct', 'ci_hi_pct', 'p', 'n', 'n_clase', 'n_vac', 'G_semanas']]
for col in ['pct', 'ci_lo_pct', 'ci_hi_pct']:
    tabla_estratos[col] = tabla_estratos[col].round(2)
tabla_estratos['p'] = tabla_estratos['p'].round(4)

print("=" * 70)
print('SCHOOL-TERM VS VACATION CONTRAST: GLOBAL AND BY SEASON (adjusted)')
print("=" * 70)
print(tabla_estratos.to_string(index=False))

tabla_estratos.to_csv(estacional_dir / 'contrastes_por_temporada.csv', index=False)
print(f"\nSaved to: {estacional_dir / 'contrastes_por_temporada.csv'}")

SCHOOL-TERM VS VACATION CONTRAST: GLOBAL AND BY SEASON (adjusted)
contaminante   estrato    pct  ci_lo_pct  ci_hi_pct      p    n  n_clase  n_vac  G_semanas
          CO    global   7.16      -4.56      20.33 0.2419 6576     5304   1272        158
          CO  invierno  10.35      -2.27      24.61 0.1119 1626     1278    348         43
          CO primavera  -0.01     -30.18      43.18 0.9995 1656     1440    216         42
          CO    verano   9.90      -0.37      21.23 0.0592 1656      948    708         42
         NO2    global   9.82       0.03      20.57 0.0492 6576     5304   1272        158
         NO2  invierno  15.00       1.57      30.21 0.0274 1626     1278    348         43
         NO2 primavera   2.81     -13.74      22.53 0.7571 1656     1440    216         42
         NO2    verano  10.66      -7.71      32.68 0.2743 1656      948    708         42
          O3    global  -7.71     -19.62       5.96 0.2550 6576     5304   1272        158
          O3  invierno -

In [6]:
# Also report the unadjusted contrast to assess the difference after adjustment
resultados_crudos = []
for c in contaminantes:
    fila = {'contaminante': c}
    global_crudo = contraste_clase(panel, f'l_{c}', con_controles=False)
    fila['global_crudo_pct'] = round(global_crudo['pct'], 1)
    for temporada in ['invierno', 'primavera', 'verano']:
        sub = panel[panel['temporada4'] == temporada]
        resultado = contraste_clase(sub, f'l_{c}', con_controles=False)
        fila[f'{temporada}_crudo_pct'] = round(resultado['pct'], 1) if resultado else np.nan
    resultados_crudos.append(fila)
tabla_crudos = pd.DataFrame(resultados_crudos)
print('\nUNADJUSTED CONTRAST, global vs within season:')
print(tabla_crudos.to_string(index=False))
tabla_crudos.to_csv(estacional_dir / 'contrastes_crudos_por_temporada.csv', index=False)
print(f"Saved to: {estacional_dir / 'contrastes_crudos_por_temporada.csv'}")


UNADJUSTED CONTRAST, global vs within season:
contaminante  global_crudo_pct  invierno_crudo_pct  primavera_crudo_pct  verano_crudo_pct
          CO              16.1                15.4                -11.6               6.7
         NO2              17.8                19.2                 -5.1              -2.4
          O3               2.2                -9.7                 -6.8              19.4
        PM10               6.8                 9.9                 11.3              -2.4
       PM2.5              10.4                 4.8                 14.9               9.1
Saved to: ../data_processed/analisis_estacional/contrastes_crudos_por_temporada.csv


In [7]:
fig, axes = plt.subplots(1, 5, figsize=(17, 5.5), sharey=False)
orden_cajas = ['invierno', 'primavera', 'verano']
colores_regimen = {0: '#3498db', 1: '#e74c3c'}

for i, c in enumerate(contaminantes):
    ax = axes[i]
    posiciones, datos, colores_caja = [], [], []
    for j, temporada in enumerate(orden_cajas):
        for k, cl in enumerate((0, 1)):
            sub = panel[(panel['temporada4'] == temporada) & (panel['clase'] == cl)][c].dropna()
            if len(sub) == 0:
                continue
            datos.append(sub.values)
            posiciones.append(j * 2.6 + k)
            colores_caja.append(colores_regimen[cl])
    cajas = ax.boxplot(datos, positions=posiciones, widths=0.8, patch_artist=True,
                       showfliers=False, medianprops={'color': 'black', 'linewidth': 1.6})
    for caja, color in zip(cajas['boxes'], colores_caja):
        caja.set_facecolor(color)
        caja.set_alpha(0.75)
    ax.set_title(c, fontsize=12, fontweight='bold')
    ax.set_xticks([j * 2.6 + 0.5 for j in range(len(orden_cajas))])
    ax.set_xticklabels([etiquetas_temporada[tp].split(' ')[0] for tp in orden_cajas], fontsize=9)
    if i == 0:
        ax.set_ylabel('Level (arrival window)', fontsize=11, fontweight='bold')

leyenda = [plt.Rectangle((0, 0), 1, 1, fc='#3498db', alpha=0.75),
           plt.Rectangle((0, 0), 1, 1, fc='#e74c3c', alpha=0.75)]
axes[0].legend(leyenda, ['Vacation', 'School term'], fontsize=9, loc='upper right')
fig.suptitle('Exploratory distributions by season and regime: substantial seasonal variation\n(boxplots omit outliers; adjusted estimates and CIs appear in section 7 and notebooks 05 and 06)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(estacional_dir / '02_boxplots_temporada_regimen.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {estacional_dir / '02_boxplots_temporada_regimen.png'}")

Plot saved to: ../data_processed/analisis_estacional/02_boxplots_temporada_regimen.png


## 6. Reversing the comparison: fixed regime, varying season

Restricting the sample to school-term dates, how much do concentrations vary between seasons? Comparing this variation with regime differences helps assess the importance of seasonal confounding.

In [8]:
solo_clase = panel[panel['clase'] == 1]
solo_vacaciones = panel[panel['clase'] == 0]

niveles_temporada_clase = solo_clase.groupby('temporada4')[contaminantes].mean().reindex(orden_temporadas)
niveles_temporada_vac = solo_vacaciones.groupby('temporada4')[contaminantes].mean().reindex(orden_temporadas)

comparativo = []
for c in contaminantes:
    niveles = niveles_temporada_clase[c].dropna()
    rango_estacional_pct = (niveles.max() / niveles.min() - 1) * 100
    contraste_invierno = tabla_estratos.query("contaminante == @c and estrato == 'invierno'")['pct']
    comparativo.append({'contaminante': c,
                        'rango_entre_temporadas_solo_clase_pct': round(rango_estacional_pct, 1),
                        'clase_vs_vac_dentro_invierno_pct': round(float(contraste_invierno.iloc[0]), 1)})
tabla_comparativo = pd.DataFrame(comparativo)

print("=" * 70)
print('WHICH VARIES MORE: SEASONAL LEVELS OR SCHOOL-TERM CONTRASTS?')
print("=" * 70)
print(tabla_comparativo.to_string(index=False))
print('\n=> Holding the regime fixed (school term), seasonal level differences are several')
print('   times larger than the within-season regime contrasts.')

tabla_comparativo.to_csv(estacional_dir / 'rango_temporadas_vs_regimen.csv', index=False)
niveles_temporada_clase.round(2).to_csv(estacional_dir / 'niveles_por_temporada_solo_clase.csv')
niveles_temporada_vac.round(2).to_csv(estacional_dir / 'niveles_por_temporada_solo_vacaciones.csv')
print(f"\nSaved to: {estacional_dir / 'rango_temporadas_vs_regimen.csv'}")

WHICH VARIES MORE: SEASONAL LEVELS OR SCHOOL-TERM CONTRASTS?
contaminante  rango_entre_temporadas_solo_clase_pct  clase_vs_vac_dentro_invierno_pct
          CO                                   98.0                              10.3
         NO2                                   84.0                              15.0
          O3                                   50.8                             -15.4
        PM10                                   54.0                              -1.1
       PM2.5                                   44.7                              -1.2

=> Holding the regime fixed (school term), seasonal level differences are several
   times larger than the within-season regime contrasts.

Saved to: ../data_processed/analisis_estacional/rango_temporadas_vs_regimen.csv


In [9]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(contaminantes))
width = 0.35
bars1 = ax.bar(x - width/2, tabla_comparativo['rango_entre_temporadas_solo_clase_pct'], width,
               label='Between seasons (school-term dates only)', color='#f39c12', alpha=0.85, edgecolor='black')
bars2 = ax.bar(x + width/2, tabla_comparativo['clase_vs_vac_dentro_invierno_pct'].abs(), width,
               label='Between regimes (winter only)', color='#3498db', alpha=0.85, edgecolor='black')

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f'{bar.get_height():.0f}%',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(contaminantes)
ax.set_xlabel('Pollutant', fontsize=12, fontweight='bold')
ax.set_ylabel('Absolute change in levels (%)', fontsize=12, fontweight='bold')
ax.set_title('Seasonal differences exceed school-term contrasts\n(absolute magnitudes)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(estacional_dir / '03_temporada_vs_regimen.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {estacional_dir / '03_temporada_vs_regimen.png'}")

Plot saved to: ../data_processed/analisis_estacional/03_temporada_vs_regimen.png


## 7. Within-day contrasts, season by season

The arrival-minus-comparison gap removes factors shared multiplicatively by both windows on the same day. Estimating it separately by season shows where the association is most apparent and how much vacation-date support each stratum provides.

Here confidence intervals are central to interpretation and are reported in full. Daily gaps vary substantially, so strata with fewer observations yield wide intervals. Additional SIMA years and school-calendar cycles could improve precision, depending on their information content and dependence structure; the familiar $1/\sqrt{n}$ rate is only a benchmark under suitable assumptions. Repeating the analysis with additional years is a natural extension. Notebook 05 reports residual diagnostics, including Durbin–Watson.

In [10]:
resultados_gap = []
for c in contaminantes:
    fila_global = contraste_clase(gaps, f'gap_{c}')
    fila_global.update({'contaminante': c, 'estrato': 'global'})
    resultados_gap.append(fila_global)
    for temporada in ['invierno', 'primavera', 'verano']:
        sub = gaps[gaps['temporada4'] == temporada]
        resultado = contraste_clase(sub, f'gap_{c}')
        if resultado is None:
            continue
        resultado.update({'contaminante': c, 'estrato': temporada})
        resultados_gap.append(resultado)

tabla_gap = pd.DataFrame(resultados_gap)[
    ['contaminante', 'estrato', 'pct', 'ci_lo_pct', 'ci_hi_pct', 'p', 'n']]
for col in ['pct', 'ci_lo_pct', 'ci_hi_pct']:
    tabla_gap[col] = tabla_gap[col].round(2)
tabla_gap['p'] = tabla_gap['p'].round(4)

print("=" * 70)
print('WITHIN-DAY GAP DIFFERENCE-IN-DIFFERENCES, GLOBAL AND BY SEASON')
print("=" * 70)
print(tabla_gap.to_string(index=False))

tabla_gap.to_csv(estacional_dir / 'gap_franjas_por_temporada.csv', index=False)
print(f"\nSaved to: {estacional_dir / 'gap_franjas_por_temporada.csv'}")

WITHIN-DAY GAP DIFFERENCE-IN-DIFFERENCES, GLOBAL AND BY SEASON
contaminante   estrato    pct  ci_lo_pct  ci_hi_pct      p    n
          CO    global   4.56       1.49       7.71 0.0033 6576
          CO  invierno   2.48      -1.69       6.83 0.2484 1626
          CO primavera   7.20       1.62      13.09 0.0108 1656
          CO    verano   5.18      -1.26      12.05 0.1172 1656
         NO2    global   1.02      -6.15       8.74 0.7867 6576
         NO2  invierno  -1.61      -8.10       5.33 0.6403 1626
         NO2 primavera  -1.04     -14.61      14.68 0.8890 1656
         NO2    verano   6.55      -9.54      25.49 0.4476 1656
          O3    global  -2.61     -12.67       8.61 0.6347 6576
          O3  invierno   0.10     -19.12      23.88 0.9929 1626
          O3 primavera -11.07     -27.51       9.11 0.2609 1656
          O3    verano   1.53     -10.54      15.22 0.8140 1656
        PM10    global  -0.54      -5.68       4.89 0.8427 6576
        PM10  invierno  -3.28      -8.58 

## 8. Sensitivity to MIMA imputation and monitoring unit

In [11]:
# Repeat the winter stratum with MIMA
print("=" * 70)
print('CHECK 1: WINTER STRATUM WITH BOTH IMPUTATION METHODS')
print("=" * 70)
panel_mima['temporada4'] = panel_mima['mes'].map({12: 'invierno', 1: 'invierno', 2: 'invierno',
                                                  3: 'primavera', 4: 'primavera', 5: 'primavera',
                                                  6: 'verano', 7: 'verano', 8: 'verano',
                                                  9: 'otono', 10: 'otono', 11: 'otono'})
for c in contaminantes:
    hibrido = contraste_clase(panel[panel['temporada4'] == 'invierno'], f'l_{c}')
    mima = contraste_clase(panel_mima[panel_mima['temporada4'] == 'invierno'], f'l_{c}')
    print(f"  {c}: hybrid {hibrido['pct']:+.1f}% (p={hibrido['p']:.3f}) | "
          f"MIMA {mima['pct']:+.1f}% (p={mima['p']:.3f})")

CHECK 1: WINTER STRATUM WITH BOTH IMPUTATION METHODS
  CO: hybrid +10.4% (p=0.112) | MIMA +9.2% (p=0.176)
  NO2: hybrid +15.0% (p=0.027) | MIMA +15.0% (p=0.028)
  O3: hybrid -15.4% (p=0.186) | MIMA -14.2% (p=0.212)
  PM10: hybrid -1.1% (p=0.932) | MIMA -0.9% (p=0.946)
  PM2.5: hybrid -1.2% (p=0.909) | MIMA -2.6% (p=0.814)


In [12]:
# Does any monitoring unit dominate the winter contrast?
print("=" * 70)
print('CHECK 2: WINTER CONTRAST BY MONITORING UNIT (NO2, unadjusted)')
print("=" * 70)
invierno = panel[panel['temporada4'] == 'invierno']
por_estacion = []
for estacion_m, sub in invierno.groupby('estacion'):
    diferencia = (np.exp(sub[sub['clase'] == 1]['l_NO2'].mean() - sub[sub['clase'] == 0]['l_NO2'].mean()) - 1) * 100
    por_estacion.append({'estacion': estacion_m, 'dif_NO2_pct': round(diferencia, 1),
                        'n_clase': int((sub['clase'] == 1).sum()), 'n_vac': int((sub['clase'] == 0).sum())})
tabla_por_estacion = pd.DataFrame(por_estacion)
print(tabla_por_estacion.to_string(index=False))
tabla_por_estacion.to_csv(estacional_dir / 'invierno_NO2_por_estacion.csv', index=False)
print(f"\nSaved to: {estacional_dir / 'invierno_NO2_por_estacion.csv'}")

CHECK 2: WINTER CONTRAST BY MONITORING UNIT (NO2, unadjusted)
 estacion  dif_NO2_pct  n_clase  n_vac
   CENTRO         19.0      213     58
 NORESTE3         15.3      213     58
NOROESTE3         16.3      213     58
   NORTE2         19.8      213     58
      SUR         24.4      213     58
SUROESTE2         20.6      213     58

Saved to: ../data_processed/analisis_estacional/invierno_NO2_por_estacion.csv


## 9. Seasonal-analysis conclusions

In [13]:
no2_crudo_global = tabla_crudos.query("contaminante == 'NO2'").iloc[0]['global_crudo_pct']
no2_crudo_invierno = tabla_crudos.query("contaminante == 'NO2'").iloc[0]['invierno_crudo_pct']
no2_aj_invierno = tabla_estratos.query("contaminante == 'NO2' and estrato == 'invierno'").iloc[0]
co_aj_invierno = tabla_estratos.query("contaminante == 'CO' and estrato == 'invierno'").iloc[0]
gap_co_global = tabla_gap.query("contaminante == 'CO' and estrato == 'global'").iloc[0]
gap_co_invierno = tabla_gap.query("contaminante == 'CO' and estrato == 'invierno'").iloc[0]
gap_co_primavera = tabla_gap.query("contaminante == 'CO' and estrato == 'primavera'").iloc[0]

print("=" * 70)
print('CONCLUSIONS (official SEP calendar)')
print("=" * 70)
def _sig(p):
    return 'significant' if p < 0.05 else 'ns'
if abs(no2_crudo_global) < 10:
    print(f"1. With the official calendar, the global unadjusted contrast is modest (NO2:")
    print(f"   {no2_crudo_global:+.0f}%): including June and July as term dates prevents school term from simply meaning")
    print(f"   cold months and partly balances the seasonal composition.")
else:
    print(f"1. The global unadjusted contrast remains large (NO2: {no2_crudo_global:+.0f}%): the unadjusted")
    print(f"   term-vacation comparison inherits seasonal composition (vacation blocks occur in")
    print(f"   particular months); this notebook therefore compares regimes WITHIN each season.")
print(f"2. Holding the regime fixed (school term), seasonal differences are much larger")
print(f"   than within-season regime differences (see comparison table).")
print(f"3. Within winter (controlling for month), the school-term CO level contrast is")
print(f"   {co_aj_invierno['pct']:+.1f}% (p = {co_aj_invierno['p']:.3f}, {_sig(co_aj_invierno['p'])}); NO2 is {no2_aj_invierno['pct']:+.1f}%")
print(f"   ({_sig(no2_aj_invierno['p'])}). Interpretation: the winter CO contrast combines activities")
print(f"   that resume in January (work and school), not school alone.")
gap_co = tabla_gap[(tabla_gap['contaminante'] == 'CO') & (tabla_gap['estrato'] != 'global')].set_index('estrato')
estrato_max = gap_co['pct'].idxmax()
print(f"4. The MORNING contrast is the within-day gap: global {gap_co_global['pct']:+.1f}%")
print(f"   (p = {gap_co_global['p']:.4f}). By season: " +
      "; ".join(f"{e} {gap_co.loc[e, 'pct']:+.1f}% ({_sig(gap_co.loc[e, 'p'])})" for e in gap_co.index) + ".")
print(f"   The stratum with the largest point estimate is {estrato_max}. Spring vacation includes")
print(f"   Easter weeks, when general traffic may fall, not only school traffic;")
print(f"   the school-term indicator can therefore capture broader city activity.")
verano = tabla_estratos[tabla_estratos['estrato'] == 'verano']
n_sig_verano = int((verano['p'] < 0.05).sum()) if len(verano) else 0
print(f"5. Summer provides a longer comparison (June-July term vs July-August vacation):")
print(f"   pollutants with significant school-term vs vacation contrasts in this stratum: {n_sig_verano} of 5.")
print(f"6. Autumn has no official vacation dates in this calendar; no contrast is estimated.")
print(f"7. The main conclusions are compared across hybrid and MIMA imputations.")

print('\nCOMPLETED')

CONCLUSIONS (official SEP calendar)
1. The global unadjusted contrast remains large (NO2: +18%): the unadjusted
   term-vacation comparison inherits seasonal composition (vacation blocks occur in
   particular months); this notebook therefore compares regimes WITHIN each season.
2. Holding the regime fixed (school term), seasonal differences are much larger
   than within-season regime differences (see comparison table).
3. Within winter (controlling for month), the school-term CO level contrast is
   +10.3% (p = 0.112, ns); NO2 is +15.0%
   (significant). Interpretation: the winter CO contrast combines activities
   that resume in January (work and school), not school alone.
4. The MORNING contrast is the within-day gap: global +4.6%
   (p = 0.0033). By season: invierno +2.5% (ns); primavera +7.2% (significant); verano +5.2% (ns).
   The stratum with the largest point estimate is primavera. Spring vacation includes
   Easter weeks, when general traffic may fall, not only school traffi